# Aula Prática 01 — Ferramentas, Ambiente e Versionamento

1. O mapa do território: o ecossistema do cientista de dados

In [2]:
import sys

print(f"Versão do Python: {sys.version}")
print(f"Executável em uso: {sys.executable}")

Versão do Python: 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Executável em uso: c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe


A saída confirma que o Git e o uv estão instalados e mostra as versões de cada um.
Registrar essas versões já é um primeiro passo de reprodutibilidade: se outra pessoa tiver um comportamento diferente, a versão da ferramenta é a primeira coisa a comparar.

3. Nasce o projeto: terminal e estrutura de pastas

O `mkdir -p` criou a pasta do projeto com três subpastas de uma vez. Cada uma tem um papel:
`data/` guarda os dados, `notebooks/` guarda as análises exploratórias e `src/` guarda o código reutilizável.
Separar as coisas desde o início evita a bagunça de arquivos soltos. Por enquanto, o projeto ainda não tem histórico nem ambiente próprio.

4. Nasce a memória do projeto: Git e o primeiro commit

O Git informa que criou um repositório vazio na pasta oculta `.git`. É nela que todo o histórico fica guardado.
A configuração de nome e e-mail não gera saída, mas define quem assina cada commit deste projeto.
A partir daqui o Git observa a pasta, mas só registra alguma coisa quando eu pedir.

O `git status` lista o `README.md` em *Untracked files*. O Git viu o arquivo novo, mas ainda não o rastreia.
Isso mostra que salvar um arquivo não é o mesmo que versioná-lo: eu preciso escolher o que entra na próxima "fotografia".

O `git add` levou o README para a *staging area* e o `git commit` registrou a primeira versão.
A saída do commit mostra o identificador curto (hash) e informa que 1 arquivo foi criado. O `git log --oneline` agora tem uma linha: o histórico começou.
O ciclo **modificar → add → commit** é o que vou repetir a cada etapa.

5. Nasce o ambiente: isolamento e o incidente do `git status`

O `uv venv` criou a pasta `.venv` com um interpretador Python próprio do projeto. A saída indica a versão do Python usada.
O `ls -a` mostra as pastas ocultas: agora o projeto tem `.git` (memória) e `.venv` (ambiente isolado).
Bibliotecas instaladas aqui não afetam outros projetos da máquina, o que evita o problema de "atualizei o pandas e o projeto antigo quebrou".

No roteiro, o `.venv/` deveria aparecer aqui como não rastreado. Na minha execução, o `git status` mostrou a área de trabalho limpa.
Isso me fez investigar o porquê na próxima célula.

As versões recentes do `uv` criam um `.gitignore` **dentro** do `.venv` com o padrão `*`, que manda o Git ignorar tudo dentro dessa pasta. Por isso o incidente não apareceu.
O problema continua real: o `venv` padrão do Python (em versões mais antigas) e o `conda` não fazem isso, e um `git add .` colocaria centenas de arquivos de bibliotecas no histórico.
Esses arquivos não são meus, mudam a cada instalação e podem ser recriados em segundos.
Além disso, a proteção do `uv` cobre só o `.venv`. Ela não protege dados (`data/`) nem segredos (`.env`). Por isso o projeto precisa do próprio `.gitignore`.

O `cat` confirma as regras gravadas e o `git status` mostra que o único arquivo novo é o próprio `.gitignore`. Agora a exclusão do `.venv/` não depende mais da ferramenta que criou o ambiente.
As regras cobrem o ambiente (`.venv/`), caches (`__pycache__/`, `.ipynb_checkpoints/`), dados (`data/`) e segredos (`*.env`).

O `.gitignore` também faz parte do projeto, então foi versionado. O log agora tem dois commits, um para cada etapa concluída.

6. O projeto ganha dependências: instalação e registro de versões

O `uv` encontrou o `.venv` da pasta e instalou o pandas **dentro dele**, junto com as dependências dele (numpy, python-dateutil e tzdata).
O `uv pip list` mostra cada biblioteca com a versão exata. Esse detalhe importa: o mesmo código pode dar resultados diferentes, ou nem rodar, em outra versão do pandas.
Por isso não basta dizer "usei pandas", é preciso dizer qual versão.

O `requirements.txt` congela as versões exatas do ambiente no formato `biblioteca==versão`.
Ele é a receita do ambiente. Com ele, qualquer pessoa recria o mesmo ambiente com `uv pip install -r requirements.txt`.
Resumo da aula: **isolamento (`.venv`) + registro de versões (`requirements.txt`) = reprodutibilidade**.
Versionamos a receita, não a cozinha.

7. Fechamento: o histórico conta a história

Lendo de baixo para cima, o log conta a aula:
1. o projeto nasceu e ganhou memória (README);
2. o ambiente virtual chegou, causou o incidente e o `.gitignore` o resolveu;
3. o projeto ganhou a primeira dependência, com a versão registrada.

Com mensagens descritivas, o histórico vira uma narrativa que qualquer pessoa consegue seguir.
Uma mensagem vaga como "ajustes" não diria nada sobre o que mudou.

8. Perguntas para consolidar

**Pergunta 1: o colega cético.** *Ambiente virtual é exagero para quem trabalha sozinho em um único projeto?*

Com um projeto só, ele pode não sentir o problema por um tempo, mas o risco existe. Quando houver dois projetos que precisam de versões diferentes da mesma biblioteca,
atualizar uma delas quebra o outro projeto em silêncio. Com um colega de equipe, sem o ambiente e o `requirements.txt` ele instala as versões mais recentes de tudo e o código pode não rodar.
E o "eu do futuro" também é outra pessoa, que não vai lembrar quais versões usou. Como criar um ambiente com o `uv` leva segundos, não vale a pena correr esse risco.

**Pergunta 2: ignorar não é proteger.** *Se eu apagar `data/vendas.csv`, o Git recupera?*

Não. A pasta `data/` está no `.gitignore`, então o Git nunca guardou nenhuma versão dela. O `README.md` pode ser recuperado com `git restore` porque está no histórico.
Ignorar um arquivo significa abrir mão da proteção do versionamento para ele. Segredos ficam de fora por segurança e o `.venv` fica de fora porque é recriável.
Já os dados valiosos precisam de outra proteção: backup e, em projetos maiores, ferramentas como DVC ou Git LFS, que guardam os dados fora do repositório e deixam no Git só uma referência.

**Pergunta 3: o teste definitivo.** *Como a colega que clonou o repositório recria o ambiente?*

1. Entra na pasta do projeto clonado: `cd projeto-vendas`;
2. Cria um ambiente novo: `uv venv`;
3. Instala as dependências registradas: `uv pip install -r requirements.txt`.

O `.venv` não viajar junto é proposital. Ele tem arquivos específicos do sistema operacional (um `.venv` do Windows não funciona no Linux) e é recriado em segundos a partir da receita.
O que precisa viajar é a informação, ou seja, o `requirements.txt`, que é leve e funciona em qualquer máquina.

9. Para casa: automatizando com `uv init` e `uv add`


Com dois comandos, o `uv` fez o que no projeto manual levou várias etapas. O `uv init` criou o projeto com `pyproject.toml`, `main.py`, `README.md` e `.python-version` (que fixa a versão do Python do projeto).
O `uv add pandas` criou o `.venv`, instalou o pandas e gerou o `uv.lock`.
(Usei `--vcs none` só para não criar um repositório Git dentro deste repositório. Sem essa opção, o `uv init` também roda o `git init` e cria um `.gitignore`.)

## Conclusão

Nesta aula montei a infraestrutura de um projeto de dados: estrutura de pastas, histórico com Git, ambiente isolado e registro de dependências.
O principal aprendizado é que reprodutibilidade não acontece por acaso. Ela depende de versionar o código e a receita do ambiente e de deixar fora do histórico o que é recriável ou sensível.

> Observação: as pastas `projeto-vendas/` e `projeto-uv/` são criadas ao executar o notebook e estão no `.gitignore` deste repositório, para não colocar um repositório Git dentro de outro.